[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/survey_analysis.ipynb)

# Psychology of Everyday Life: Survey Analysis

This notebook provides tools for analyzing the class survey data from the Psychology of Everyday Life lab. You'll explore relationships between everyday behaviors and attitudes using statistical tests and visualizations.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style for cleaner visuals
sns.set_theme(style='whitegrid')
%matplotlib inline

## Loading the Survey Data

The class survey responses are stored in a Google Sheet. To load them:

1. Open the [survey responses spreadsheet](https://docs.google.com/spreadsheets/d/1MvZoEIU5OdAOVtUTw8QoYQdoz1HcOhInSqeZcE0wQgg/edit?usp=sharing).
2. Go to **File > Share > Publish to web**.
3. Under "Link", select the sheet and choose **Comma-separated values (.csv)** as the format.
4. Click **Publish** and copy the URL.
5. Paste that URL into the `data_url` variable in the cell below.

Alternatively, download the sheet as CSV (**File > Download > Comma-separated values**), upload to Colab, and use `pd.read_csv('your_file.csv')`.

**Note:** Google Forms uses the full question text as column headers. The cell below renames them to short, code-friendly names.

In [ ]:
# Paste your published Google Sheet CSV URL here
# (See instructions above for how to get this URL)
data_url = 'YOUR_PUBLISHED_CSV_URL_HERE'

df = pd.read_csv(data_url)

# Rename columns from Google Forms' full question text to short names
# Adjust the mapping below if your form's question text differs
column_mapping = {
    'Timestamp': 'timestamp',
    'How many hours of sleep do you typically get per night?': 'sleep_hours',
    'On a scale of 1-10, how stressed do you generally feel?': 'stress_level',
    'On a scale of 1-10, how happy do you generally feel?': 'happiness',
    'How many hours per day do you spend on screens (outside of schoolwork)?': 'screen_time',
    'How many days per week do you exercise?': 'exercise_frequency',
    'How many caffeinated beverages do you consume per day?': 'caffeine_intake',
    'How many hours per week do you spend studying (outside of class)?': 'study_hours',
    'On a scale of 1-10, how socially active are you?': 'social_activity',
}

# Apply renaming -- uses fuzzy matching in case question text differs slightly
renamed = {}
for old_col in df.columns:
    for pattern, new_name in column_mapping.items():
        if pattern.lower() in old_col.lower() or old_col.lower() in pattern.lower():
            renamed[old_col] = new_name
            break
    else:
        # Keep original name if no match found
        renamed[old_col] = old_col.lower().replace(' ', '_')

df = df.rename(columns=renamed)

# Drop the timestamp column (not needed for analysis)
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

print(f'Loaded {len(df)} responses with {len(df.columns)} columns.')
print(f'Columns: {list(df.columns)}')
df.head()

## Explore the Data

In [ ]:
# Get a quick overview of the dataset
print('=== First Few Rows ===')
display(df.head())

print('\n=== Column Types and Non-Null Counts ===')
df.info()

print('\n=== Summary Statistics ===')
df.describe()

In [ ]:
# Correlation heatmap of all numeric variables
# This gives a bird's-eye view of which variables are related

numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 8))
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Survey Variables')
plt.tight_layout()
plt.show()

## Statistical Tests

Which test should you use?

| Scenario | Test |
|-|-|
| Compare means of two groups | Independent-samples t-test |
| Relationship between two continuous variables | Pearson correlation |
| Association between two categorical variables | Chi-square test |

Below are examples of each. Modify the column names to match your survey questions.

In [ ]:
# Independent-samples t-test
# Example: Do people who sleep more (>= 7 hrs) report different stress levels
# than people who sleep less (< 7 hrs)?

# Split into two groups based on a median split (adjust as needed)
high_sleep = df[df['sleep_hours'] >= 7]['stress_level']
low_sleep = df[df['sleep_hours'] < 7]['stress_level']

# Run the t-test
t_stat, p_value = stats.ttest_ind(high_sleep.dropna(), low_sleep.dropna())

print(f'High sleep group: n={len(high_sleep)}, mean={high_sleep.mean():.2f}, sd={high_sleep.std():.2f}')
print(f'Low sleep group:  n={len(low_sleep)}, mean={low_sleep.mean():.2f}, sd={low_sleep.std():.2f}')
print(f'\nt-statistic = {t_stat:.3f}')
print(f'p-value     = {p_value:.4f}')
print(f'\nThe difference is {"statistically significant" if p_value < 0.05 else "not statistically significant"} at alpha = 0.05.')

In [ ]:
# Pearson correlation
# Example: Is there a relationship between screen time and happiness?

# Drop rows where either variable is missing
clean = df[['screen_time', 'happiness']].dropna()

r, p_value = stats.pearsonr(clean['screen_time'], clean['happiness'])

print(f'Pearson r = {r:.3f}')
print(f'p-value   = {p_value:.4f}')
print(f'\nThis indicates a {"positive" if r > 0 else "negative"} correlation.')
print(f'The correlation is {"statistically significant" if p_value < 0.05 else "not statistically significant"} at alpha = 0.05.')

In [ ]:
# Chi-square test of independence
# Example: Is exercise frequency associated with caffeine intake category?

# First, create categorical bins if your variables are continuous
# (Skip this step if your variables are already categorical)
df['exercise_cat'] = pd.cut(df['exercise_frequency'], bins=3, labels=['Low', 'Medium', 'High'])
df['caffeine_cat'] = pd.cut(df['caffeine_intake'], bins=3, labels=['Low', 'Medium', 'High'])

# Build a contingency table
contingency = pd.crosstab(df['exercise_cat'], df['caffeine_cat'])
print('Contingency Table:')
display(contingency)

# Run the chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f'\nChi-square statistic = {chi2:.3f}')
print(f'Degrees of freedom   = {dof}')
print(f'p-value              = {p_value:.4f}')
print(f'\nThe association is {"statistically significant" if p_value < 0.05 else "not statistically significant"} at alpha = 0.05.')

## Visualization

In [ ]:
# Scatter plot with regression line
# Example: Relationship between study hours and happiness

plt.figure(figsize=(8, 6))
sns.regplot(data=df, x='study_hours', y='happiness',
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.xlabel('Study Hours per Week')
plt.ylabel('Happiness Rating')
plt.title('Study Hours vs. Happiness')
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart comparing means across groups
# Example: Mean stress level and happiness by social activity level

# Create a categorical grouping variable
df['social_group'] = pd.cut(df['social_activity'], bins=3, labels=['Low', 'Medium', 'High'])

# Compute group means for the variables of interest
group_means = df.groupby('social_group')[['stress_level', 'happiness']].mean()

# Plot
ax = group_means.plot(kind='bar', figsize=(8, 6), colormap='Set2', edgecolor='black')
plt.xlabel('Social Activity Level')
plt.ylabel('Mean Rating')
plt.title('Stress and Happiness by Social Activity Level')
plt.xticks(rotation=0)
plt.legend(title='Measure')
plt.tight_layout()
plt.show()

## Interpreting Your Results

A few things to keep in mind:

- **Correlation is not causation.** Even a strong, significant correlation between two variables does not tell us that one *causes* the other. There may be confounding variables ("third variables") that explain the relationship.
- **Sample size matters.** With a small class, you may not have enough statistical power to detect real effects. Non-significant results don't necessarily mean there's no relationship -- you may just lack the power to detect it.
- **Multiple comparisons.** If you run many tests, some will be "significant" by chance alone. Be cautious about drawing strong conclusions from a single p-value.
- **Self-report limitations.** Survey data relies on honest, accurate self-report. People may misremember, round, or give socially desirable answers.
- **Think about confounds.** Before concluding that variable A is related to variable B, ask: what other variables might explain this pattern?

### Keep Exploring!

The analyses above are just starting points. Try different combinations of variables, create new visualizations, or run additional tests. You're encouraged to use generative AI tools (e.g., ChatGPT, Claude) to help you brainstorm analyses, debug code, or interpret results -- just make sure you understand what the code is doing and can explain your findings in your own words.